In [ ]:
import os
import numpy as np
import cv2
from pycocotools.coco import COCO
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
def generate_colored_mask(json_path, output_dir, img_size=(128, 128)):
    os.makedirs(output_dir, exist_ok=True)

    coco=COCO(json_path)
    cat_ids=coco.getCatIds()
    categories= coco.loadCats(cat_ids)

    colormap= plt.get_cmap("tab10", len(cat_ids))

    id_to_color = {
        cat['id']: tuple((np.array(colormap(i)[:3]) * 255).astype(np.uint8))
        for i, cat in enumerate(categories)
    }

    for img_id in coco.getImgIds():
        #reteive size of the images to create black images with the same size
        img_info=coco.loadImgs(img_id)[0]
        img_w, img_h= img_info['width'], img_info['height']

        mask_rgb= np.zeros((img_h, img_w, 3), dtype= np.uint8)

        ann_ids= coco.getAnnIds(imgIds= img_id)
        anns= coco.loadAnns(ann_ids)

        for ann in anns:
            cat_id = ann['category_id']
            color = id_to_color[cat_id]

            ann_mask = coco.annToMask(ann)
            for c in range(3):
                mask_rgb[:, :, c] = np.where(ann_mask == 1, color[c], mask_rgb[:, :, c])

        mask_rgb = cv2.resize(mask_rgb, img_size)

        mask_filename= img_info['file_name'].replace('.jpg', '_segmask.png')
        mask_path= os.path.join(output_dir, mask_filename)

        Image.fromarray(mask_rgb).save(mask_path)

        print(f"Saved segmentation mask: {mask_path}")


generate_colored_mask(
    json_path='/Users/mac/Desktop/projet/dataset_COCO/valid/_annotations.coco.json',
    #image_dir='/Users/mac/Desktop/projet/dataset_COCO/train/train_img',
    output_dir='/Users/mac/Desktop/projet/dataset_COCO/valid/valid_mask',
    img_size=(128,128)
)